# N1 · 给候选研究方向打分对比 (Score Candidate Directions)

> 配套 L1 · **真实科研动作**: 列出你脑子里真实存在的2个候选研究方向(哪怕只是模糊念头),
> 用四维打分框架(兴趣/实验室积累/资助趋势/职业规划)真的打一次分, 对比出结论。

不需要虚构材料 —— 换成你自己现在真的在纠结的两个方向去跑一遍。

In [1]:
import sys
from pathlib import Path
SRC = Path.cwd().parent / "src"
sys.path.insert(0, str(SRC))
import direction_scorer as ds
print("四个打分维度:", [ds.DIMENSIONS[k][0] for k in ds.DIMENSIONS])

四个打分维度: ['兴趣真实度', '实验室积累契合度', '资助与关注度趋势', '职业规划契合度']


## 1. 建两个候选, 逐维度打分(必须写依据, 空口打分等于没打分)

In [2]:
a = ds.blank_candidate("方向A: 长上下文推理效率")
a["scores"]["interest"] = {"score": 4, "note": "读过的相关论文都主动做了笔记, 是真兴趣"}
a["scores"]["lab_fit"] = {"score": 5, "note": "long-context专题的代码和数据直接能复用"}
a["scores"]["funding_fit"] = {"score": 3, "note": "关注度稳定但不算上升期, 竞争者不少"}
a["scores"]["career_fit"] = {"score": 4, "note": "工业界效率方向岗位需求持续存在"}

b = ds.blank_candidate("方向B: 多模态对齐评测")
b["scores"]["interest"] = {"score": 2, "note": "是最近才因为热度关注, 还没深入读过几篇"}
b["scores"]["lab_fit"] = {"score": 1, "note": "实验室没有多模态数据/算力积累, 要从零搭"}
b["scores"]["funding_fit"] = {"score": 5, "note": "明显上升期, 顶会track专门扩了"}
b["scores"]["career_fit"] = {"score": 3, "note": "方向新, 但和自己规划的NLP路线略有偏离"}

print(ds.render([a, b]))

=== 候选研究方向对比 ===

1. 方向A: 长上下文推理效率  (总分 16/20, 最弱项: 资助与关注度趋势)
   兴趣真实度: 4分 —— 读过的相关论文都主动做了笔记, 是真兴趣
   实验室积累契合度: 5分 —— long-context专题的代码和数据直接能复用
   资助与关注度趋势: 3分 —— 关注度稳定但不算上升期, 竞争者不少
   职业规划契合度: 4分 —— 工业界效率方向岗位需求持续存在

2. 方向B: 多模态对齐评测  (总分 11/20, 最弱项: 实验室积累契合度)
   兴趣真实度: 2分 —— 是最近才因为热度关注, 还没深入读过几篇
   实验室积累契合度: 1分 —— 实验室没有多模态数据/算力积累, 要从零搭
   资助与关注度趋势: 5分 —— 明显上升期, 顶会track专门扩了
   职业规划契合度: 3分 —— 方向新, 但和自己规划的NLP路线略有偏离


## 2. 打分完整性自检(每维度必须有分数+依据)

In [3]:
for c in [a, b]:
    chk = ds.audit(c)
    status = "✅ 完整" if chk["ready"] else "⚠ " + "; ".join(chk["issues"])
    print(f"{c['name']}: {status}")

方向A: 长上下文推理效率: ✅ 完整
方向B: 多模态对齐评测: ✅ 完整


## 3. 反面: 只打分不写依据

In [4]:
bad = ds.blank_candidate("方向C: 敷衍打分示例")
bad["scores"]["interest"] = {"score": 5, "note": ""}
for i in ds.audit(bad)["issues"]:
    print("⚠", i)
print("\n→ 只填数字不写理由, 等于没打分 —— 半年后你自己都不记得当初为什么打这个分。")

⚠ 「兴趣真实度」打了 5 分却没写依据 —— 空口打分等于没打分
⚠ 「实验室积累契合度」缺分数或分数越界(应为1-5): 当前 0
⚠ 「资助与关注度趋势」缺分数或分数越界(应为1-5): 当前 0
⚠ 「职业规划契合度」缺分数或分数越界(应为1-5): 当前 0

→ 只填数字不写理由, 等于没打分 —— 半年后你自己都不记得当初为什么打这个分。


## 4. 总分最高 ≠ 直接选它

方向A总分更高, 但方向B的资助趋势明显更强。这正是L2「项目级可行性评估」要接着回答的问题:
方向A的lab_fit优势能否弥补funding_fit的中庸, 需要结合更长尺度的可行性判断, 不能只看总分表。

In [5]:
ranked = ds.compare([a, b])
print("排序结果(仅供参考, 不是最终决定):")
for i, c in enumerate(ranked, 1):
    print(f"  {i}. {c['name']} (总分 {ds.total(c)}, 最弱项: {c['_weakest']})")
print("\n下一步: 把总分靠前的1-2个候选, 拿去 N2 之前先走一遍 templates/feasibility-worksheet.md (L2)。")

排序结果(仅供参考, 不是最终决定):
  1. 方向A: 长上下文推理效率 (总分 16, 最弱项: 资助与关注度趋势)
  2. 方向B: 多模态对齐评测 (总分 11, 最弱项: 实验室积累契合度)

下一步: 把总分靠前的1-2个候选, 拿去 N2 之前先走一遍 templates/feasibility-worksheet.md (L2)。


## 5. 反思

你刚把"自己想个方向"这句空话, 变成了一份可比较、可留档的候选清单。带走:
- 兴趣/实验室积累/资助趋势/职业规划四维缺一不可, 只看兴趣或只追热点都会踩坑。
- 打分必须带依据, 不然半年后自己都解释不清当初的判断。
- 总分只是参考, 最终决定还要过L2的项目级可行性评估这一关。

下一步: 去 L2 讲义, 对总分靠前的候选跑一遍 `templates/feasibility-worksheet.md`。